# 00 · Data inventory

What the Kaggle BDB 2027 competition data contains, what units it uses, and how well the Combine tracking agrees with the official results. Outputs are aggregates only (no raw rows), per the competition data terms.

Findings feed `config/data_dictionary.md` and the Combine loader.

In [1]:
from pathlib import Path
import numpy as np
import polars as pl

from eeg.config import PROJECT_ROOT

pl.Config.set_tbl_rows(80); pl.Config.set_tbl_cols(30); pl.Config.set_tbl_width_chars(200)
D = PROJECT_ROOT / 'data/raw/kaggle/nfl-big-data-bowl-2027'
read = lambda name, **kw: pl.read_csv(D / name, null_values='NA', **kw)
scan = lambda name, **kw: pl.scan_csv(D / name, null_values='NA', **kw)

## 1. Files

In [2]:
inv = []
for p in sorted(D.glob('*.csv')):
    lf = scan(p.name)
    inv.append((p.name, round(p.stat().st_size / 1e6, 1), lf.select(pl.len()).collect().item(), len(lf.collect_schema())))
pl.DataFrame(inv, schema=['file', 'MB', 'rows', 'cols'], orient='row')

file,MB,rows,cols
str,f64,i64,i64
"""combine_results.csv""",0.0,510,18
"""combine_tracking.csv""",83.8,463189,14
"""game_tracking_2023.csv""",335.5,3665443,12
"""game_tracking_2024.csv""",718.0,7827944,12
"""game_tracking_2025.csv""",1022.6,11160903,12
"""games.csv""",0.0,1002,7
"""player_career_successes.csv""",0.0,510,9
"""player_play.csv""",146.9,314197,64
"""players.csv""",0.0,510,10


## 2. Combine results

One row per prospect. Heights/arms/hands/wingspan in inches, weight in lb, vertical/broad in inches, times in s.

In [3]:
cr = read('combine_results.csv')
print(cr.shape, '| nfl_id unique:', cr['nfl_id'].is_unique().all())
cr.group_by('draft_year', 'combine_position').len().pivot(on='combine_position', index='draft_year', values='len').sort('draft_year')

(510, 18) | nfl_id unique: True


draft_year,OL,TE,DB,WR,DL
i64,u32,u32,u32,u32,u32
2023,37,12,35,39,37
2024,46,13,50,31,37
2025,38,17,37,37,44


In [4]:
drills = ['ten_yd_split', 'forty', 'vertical', 'broad_jump', 'three_cone', 'short_shuttle', 'bench_reps']
cov = cr.group_by('combine_position').agg(pl.len().alias('n'), *[pl.col(d).is_not_null().mean().round(2).alias(d) for d in drills])
cov.sort('combine_position')

combine_position,n,ten_yd_split,forty,vertical,broad_jump,three_cone,short_shuttle,bench_reps
str,u32,f64,f64,f64,f64,f64,f64,f64
"""DB""",122,0.85,0.85,0.86,0.8,0.2,0.27,0.37
"""DL""",118,0.8,0.8,0.86,0.82,0.36,0.39,0.46
"""OL""",121,0.79,0.79,0.83,0.81,0.5,0.53,0.44
"""TE""",42,0.81,0.81,0.83,0.83,0.43,0.48,0.29
"""WR""",107,0.91,0.91,0.9,0.89,0.31,0.36,0.26


In [5]:
cr.drop('nfl_id', 'draft_year', 'combine_position').describe()

statistic,combine_height,combine_weight,hand_size,arm_length,wing_span,ten_yd_split,forty,vertical,broad_jump,three_cone,short_shuttle,bench_reps,ngs_athleticism_score,ngs_college_production_score,ngs_final_score
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",510.0,510.0,510.0,510.0,510.0,425.0,424.0,438.0,423.0,180.0,201.0,192.0,510.0,509.0,509.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,85.0,86.0,72.0,87.0,330.0,309.0,318.0,0.0,1.0,1.0
"""mean""",74.579657,248.380392,9.571814,32.410784,78.465931,1.632,4.733514,33.810502,118.1513,7.310611,4.466318,21.71875,72.454902,67.475442,70.333988
"""std""",2.628102,53.254702,0.602178,1.419286,3.273243,0.115573,0.309449,4.141179,9.064842,0.390453,0.254316,5.718901,10.612339,8.317156,7.530414
"""min""",68.125,154.0,8.0,28.5,69.0,1.46,4.28,23.0,93.0,6.57,3.93,8.0,51.0,50.0,52.0
"""25%""",72.625,198.0,9.125,31.375,76.25,1.54,4.47,31.0,111.0,7.0,4.28,17.0,65.0,61.0,65.0
"""50%""",75.0,246.0,9.5,32.375,78.5,1.59,4.62,34.0,120.0,7.3,4.45,22.0,73.0,66.0,70.0
"""75%""",76.5,304.0,10.0,33.5,80.875,1.74,5.01,37.0,125.0,7.62,4.68,26.0,80.0,73.0,75.0
"""max""",80.625,374.0,11.625,36.875,88.25,1.96,5.48,44.0,138.0,8.19,5.03,39.0,99.0,99.0,94.0


In [6]:
nd = cr.select(pl.sum_horizontal([pl.col(d).is_not_null() for d in drills if d != 'ten_yd_split']).alias('n_drills'))
nd['n_drills'].value_counts().sort('n_drills')

n_drills,count
u32,u32
0,27
1,24
2,37
3,149
4,118
5,89
6,66


## 3. ID linkage (`nfl_id`)

In [7]:
pl_ = read('players.csv'); cs = read('player_career_successes.csv')
ct = read('combine_tracking.csv', try_parse_dates=True)
ids = set(cr['nfl_id'])
game_ids = set(pl.concat([scan(f'game_tracking_{y}.csv').select('nfl_id').unique().collect() for y in (2023, 2024, 2025)])['nfl_id'])
print('players.csv == combine_results:', set(pl_['nfl_id']) == ids)
print('career_successes == combine_results:', set(cs['nfl_id']) == ids)
print('combine_tracking subset:', set(ct['nfl_id']) <= ids, '| prospects with any combine tracking:', len(set(ct['nfl_id']) & ids))
print('prospects appearing in game tracking:', len(game_ids & ids), '/', len(ids), '| other ids in game tracking:', len(game_ids - ids))

players.csv == combine_results: True
career_successes == combine_results: True
combine_tracking subset: True | prospects with any combine tracking: 510
prospects appearing in game tracking: 497 / 510 | other ids in game tracking: 0


In [8]:
# combine_position vs NFL position (players.csv)
cr.join(pl_, on='nfl_id').group_by('combine_position', 'nfl_position').len().sort('combine_position', 'len', descending=[False, True])

combine_position,nfl_position,len
str,str,u32
"""DB""","""CB""",60
"""DB""","""SS""",28
"""DB""","""FS""",25
"""DB""","""DB""",5
"""DB""","""ILB""",2
"""DB""","""MLB""",1
"""DB""","""RB""",1
"""DL""","""DT""",47
"""DL""","""DE""",36


## 4. Combine tracking

10 Hz. `x`/`y` in yards (field coordinates), `s` in yd/s, `a` in yd/s².

In [9]:
print(ct['entity_type'].value_counts())
ctp = ct.filter(pl.col('entity_type') == 'PLAYER').sort('time')
REP = ['nfl_id', 'event_id', 'attempt']
dt = ctp.group_by(REP).agg(pl.col('time').diff().dt.total_milliseconds().alias('dt')).explode('dt').drop_nulls()
print('frame interval (ms):', dt['dt'].value_counts().sort('count', descending=True).head(3).rows())

shape: (2, 2)
┌─────────────┬────────┐
│ entity_type ┆ count  │
│ ---         ┆ ---    │
│ str         ┆ u32    │
╞═════════════╪════════╡
│ BALL        ┆ 32095  │
│ PLAYER      ┆ 431094 │
└─────────────┴────────┘
frame interval (ms): [(100, 424784)]


In [10]:
ctp.group_by('drill_type').agg(pl.col('nfl_id').n_unique().alias('players'), pl.struct(REP).n_unique().alias('reps'), pl.col('drill_name').n_unique().alias('drill_names')).sort('drill_type')

drill_type,players,reps,drill_names
str,u32,u32,u32
"""FORTY_YARD_DASH""",418,793,1
"""SHORT_SHUTTLE""",170,223,1
"""SKILL_DRILLS_DB""",122,1170,9
"""SKILL_DRILLS_DL""",114,1007,11
"""SKILL_DRILLS_LB""",4,37,6
"""SKILL_DRILLS_OL""",121,910,8
"""SKILL_DRILLS_TE""",42,438,10
"""SKILL_DRILLS_WR""",107,1529,15
"""THREE_CONE_DRILL""",160,203,1


### 4a. Is `dis` consistent with `x`/`y` and `s`?

In [11]:
chk = ctp.with_columns(((pl.col('x').diff() ** 2 + pl.col('y').diff() ** 2).sqrt()).over(REP).alias('step')).filter(pl.col('step') > 0.05)
chk.group_by('drill_type').agg(
    (pl.col('dis') / pl.col('step')).median().round(3).alias('dis / xy_step'),
    (pl.col('s') * 0.1 / pl.col('step')).median().round(3).alias('s·dt / xy_step'),
).sort('drill_type')

drill_type,dis / xy_step,s·dt / xy_step
str,f64,f64
"""FORTY_YARD_DASH""",0.899,1.007
"""SHORT_SHUTTLE""",0.298,0.986
"""SKILL_DRILLS_DB""",0.525,1.004
"""SKILL_DRILLS_DL""",0.336,1.001
"""SKILL_DRILLS_LB""",0.416,1.004
"""SKILL_DRILLS_OL""",0.249,1.007
"""SKILL_DRILLS_TE""",0.586,1.003
"""SKILL_DRILLS_WR""",0.595,1.003
"""THREE_CONE_DRILL""",0.265,0.989


### 4b. Speed glitches

Fastest NFL players top out near 12 yd/s (~24.5 mph). Frames above 13 yd/s are treated as tracking glitches.

In [12]:
ctp.group_by('drill_type').agg(pl.col('s').quantile(0.999).round(2).alias('s_p99.9'), pl.col('s').max().round(2).alias('s_max'), (pl.col('s') > 13).sum().alias('frames_s>13'), pl.len().alias('frames')).sort('drill_type')

drill_type,s_p99.9,s_max,frames_s>13,frames
str,f64,f64,u32,u32
"""FORTY_YARD_DASH""",11.64,11.85,0,41500
"""SHORT_SHUTTLE""",6.47,6.87,0,11690
"""SKILL_DRILLS_DB""",9.7,10.26,0,94623
"""SKILL_DRILLS_DL""",8.97,9.29,0,77428
"""SKILL_DRILLS_LB""",8.61,8.63,0,3263
"""SKILL_DRILLS_OL""",8.16,8.49,0,61129
"""SKILL_DRILLS_TE""",9.6,9.79,0,29102
"""SKILL_DRILLS_WR""",10.47,10.98,0,96408
"""THREE_CONE_DRILL""",7.36,8.11,0,15951


### 4c. 40-yard dash: tracking vs official

Crossing times are interpolated on absolute field lines: start line x = 10, 10-yd mark x = 20, finish x = 50.

In [13]:
def cross(t, x, d):
    if x[0] >= d or x[-1] < d:
        return np.nan
    i = int(np.argmax(x >= d))
    return t[i - 1] + (d - x[i - 1]) / (x[i] - x[i - 1]) * (t[i] - t[i - 1])

rows = []
for (nid, ev, att, yr), r in ctp.filter(pl.col('drill_type') == 'FORTY_YARD_DASH').group_by(['nfl_id', 'event_id', 'attempt', 'draft_year']):
    t = (r['time'] - r['time'][0]).dt.total_milliseconds().to_numpy() / 1000
    x = r['x'].to_numpy()
    rows.append((nid, yr, x[0], x[-1], r['s'][0], cross(t, x, 20), cross(t, x, 50)))
f40 = pl.DataFrame(rows, schema=['nfl_id', 'yr', 'x_first', 'x_last', 's_first', 't20', 't50'], orient='row').fill_nan(None)
print('reps:', f40.height, '| reaching x=50:', f40['t50'].is_not_null().sum())
f40.select(pl.col('x_first', 'x_last', 's_first').quantile(0.01).name.suffix('_p01'), pl.col('x_first', 'x_last', 's_first').median().name.suffix('_med'), pl.col('x_first', 'x_last', 's_first').quantile(0.99).name.suffix('_p99'))

reps: 793 | reaching x=50: 793


x_first_p01,x_last_p01,s_first_p01,x_first_med,x_last_med,s_first_med,x_first_p99,x_last_p99,s_first_p99
f64,f64,f64,f64,f64,f64,f64,f64,f64
9.434626,50.492279,0.297382,10.66877,51.066137,0.63,11.833333,52.166667,1.38


In [14]:
best = (f40.with_columns((pl.col('t50') - pl.col('t20')).alias('fly30_trk'))
        .group_by('nfl_id', 'yr').agg(pl.col('fly30_trk').min(), pl.col('t50').min(), pl.col('t20').min())
        .join(cr.select('nfl_id', 'forty', 'ten_yd_split'), on='nfl_id')
        .with_columns((pl.col('forty') - pl.col('ten_yd_split')).alias('fly30_off')))
best.group_by('yr').agg(
    pl.len().alias('players'),
    (pl.col('fly30_trk') - pl.col('fly30_off')).median().round(3).alias('fly30 bias (s)'),
    (pl.col('fly30_trk') - pl.col('fly30_off')).std().round(3).alias('fly30 sd (s)'),
    pl.corr('fly30_trk', 'fly30_off').round(3).alias('fly30 r'),
    (pl.col('t50') - pl.col('forty')).median().round(3).alias('window-start lead (s)'),
    (pl.col('t20') - pl.col('ten_yd_split')).median().round(3).alias('lead via 10-yd (s)'),
).sort('yr')

yr,players,fly30 bias (s),fly30 sd (s),fly30 r,window-start lead (s),lead via 10-yd (s)
i64,u32,f64,f64,f64,f64,f64
2023,138,-0.026,0.052,0.971,0.175,0.205
2024,147,-0.024,0.034,0.988,0.166,0.19
2025,133,-0.027,0.042,0.981,0.293,0.324


In [15]:
print('official forty but no tracked 40:', cr.filter(pl.col('forty').is_not_null() & ~pl.col('nfl_id').is_in(best['nfl_id'].implode())).height)
print('tracked 40 but no official forty:', best.filter(pl.col('forty').is_null()).height)
print('official forty but no ten_yd_split:', cr.filter(pl.col('forty').is_not_null() & pl.col('ten_yd_split').is_null()).height)

official forty but no tracked 40: 6
tracked 40 but no official forty: 0
official forty but no ten_yd_split: 0


### 4d. Shuttle & 3-cone: tracked rep duration vs official

In [16]:
cod = {'SHORT_SHUTTLE': 'short_shuttle', 'THREE_CONE_DRILL': 'three_cone'}
out = []
for drill, col in cod.items():
    reps = (ctp.filter(pl.col('drill_type') == drill).group_by(REP)
            .agg(((pl.col('time').max() - pl.col('time').min()).dt.total_milliseconds() / 1000).alias('dur'), pl.col('s').max().alias('smax')))
    b = reps.group_by('nfl_id').agg(pl.col('dur').min(), pl.col('smax').max(), pl.len().alias('reps')).join(cr.select('nfl_id', col), on='nfl_id', how='full', coalesce=True)
    out.append((drill, b.filter(pl.col('dur').is_not_null()).height, b.filter(pl.col(col).is_not_null() & pl.col('dur').is_null()).height,
                b.filter(pl.col(col).is_null() & pl.col('dur').is_not_null()).height,
                round((b['dur'] - b[col]).median(), 3), round(b.select(pl.corr('dur', col)).item(), 3), round(b['smax'].median(), 2)))
pl.DataFrame(out, schema=['drill', 'tracked players', 'official only', 'tracked only', 'dur - official (med s)', 'r', 'peak s (med yd/s)'], orient='row')

drill,tracked players,official only,tracked only,dur - official (med s),r,peak s (med yd/s)
str,i64,i64,i64,f64,f64,f64
"""SHORT_SHUTTLE""",170,41,10,0.47,0.154,5.86
"""THREE_CONE_DRILL""",160,30,10,0.38,0.269,6.65


## 5. Game tracking (Phase 4)

In [17]:
g = []
for y in (2023, 2024, 2025):
    lf = scan(f'game_tracking_{y}.csv')
    g.append(lf.select(pl.lit(y).alias('file'), pl.len().alias('frames'), pl.struct('game_id', 'play_id').n_unique().alias('plays'), pl.col('game_id').n_unique().alias('games'),
                       pl.col('nfl_id').n_unique().alias('players'), pl.col('s').quantile(0.999).alias('s_p99.9'), pl.col('s').max().alias('s_max')).collect())
pl.concat(g)

file,frames,plays,games,players,s_p99.9,s_max
i32,u32,u32,u32,u32,f64,f64
2023,3665443,31690,333,156,9.48,12.67
2024,7827944,40331,334,318,9.4,13.87
2025,11160903,42544,334,448,9.32,16.88


In [18]:
games = read('games.csv')
print(games.group_by('season', 'season_type').len().sort('season', 'season_type'))
pp = scan('player_play.csv')
print('player_play columns:', len(pp.collect_schema()), '| rows:', pp.select(pl.len()).collect().item())

shape: (9, 3)
┌────────┬─────────────┬─────┐
│ season ┆ season_type ┆ len │
│ ---    ┆ ---         ┆ --- │
│ i64    ┆ str         ┆ u32 │
╞════════╪═════════════╪═════╡
│ 2023   ┆ POST        ┆ 13  │
│ 2023   ┆ PRE         ┆ 49  │
│ 2023   ┆ REG         ┆ 272 │
│ 2024   ┆ POST        ┆ 13  │
│ 2024   ┆ PRE         ┆ 49  │
│ 2024   ┆ REG         ┆ 272 │
│ 2025   ┆ POST        ┆ 13  │
│ 2025   ┆ PRE         ┆ 49  │
│ 2025   ┆ REG         ┆ 272 │
└────────┴─────────────┴─────┘
player_play columns: 64 | rows: 314197


## 6. Outcomes (`player_career_successes.csv`)

In [19]:
cs.drop('nfl_id').describe()

statistic,career_offensive_snaps,career_defensive_snaps,career_special_teams_snaps,career_games_active,career_games_started,ap_all_pro_1st_team,ap_all_pro_2nd_team,pro_bowl_original_ballot
str,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",510.0,510.0,510.0,510.0,510.0,510.0,510.0,510.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",327.415686,234.307843,113.503922,21.721569,7.852941,0.011765,0.003922,0.027451
"""std""",632.400084,492.121894,156.194148,16.285063,11.466015,0.107931,0.062561,0.19631
"""min""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""25%""",0.0,0.0,5.0,6.0,0.0,0.0,0.0,0.0
"""50%""",0.0,0.0,53.0,17.0,2.0,0.0,0.0,0.0
"""75%""",362.0,222.0,148.0,34.0,12.0,0.0,0.0,0.0
"""max""",3303.0,2666.0,957.0,51.0,50.0,1.0,1.0,2.0


In [20]:
pl_.select(pl.col('draft_round').is_null().sum().alias('undrafted'), pl.len().alias('prospects'))

undrafted,prospects
u32,u32
127,510


## 7. Summary

- **Population:** 510 prospects (draft years 2023-25), already labelled with our 5 groups in `combine_position`, so no position mapping is needed. `nfl_id` is unique and links every file. 127 went undrafted. 497 appear in the 2023-25 game tracking.
- **Anthropometrics are complete** for all 510: height, weight, arm length, hand size, wingspan. Bench ROM can use measured arm length; no imputation.
- **Drill coverage:** 40 / 10-yd split / vertical / broad at about 80-90% per group; 3-cone, shuttle and bench only 20-53%. 422 of 510 have ≥ 3 of the 6 drills.
- **Combine tracking:** 10 Hz exactly; `x`/`y` in yards on field coordinates; `s` in yd/s (matches the x/y step to about 1%). No frames above 13 yd/s. `BALL` rows (WR/TE skill drills) must be filtered.
- **`dis` is unreliable:** 0.25-0.90 × the x/y step depending on drill. Derive distance from `x`/`y` (or `s`) instead.
- **40-yard dash:** 418 prospects / 793 reps, all reaching the finish line (x = 50). On absolute field lines, the tracked flying 30 (x 20 → 50) matches official `forty − ten_yd_split` with bias −0.025 s, sd 0.03-0.05 s, r = 0.97-0.99. **The tracking can be trusted for the sprint fit.** Each rep window starts 0.17-0.29 s before the official clock, and the lead differs by year, so the sprint fit should treat the start time as a free parameter (or use x = 10 crossing) rather than the first frame.
- **Shuttle / 3-cone:** 170 / 160 tracked prospects. The windows run about 0.4-0.5 s longer than the official times and correlate weakly with them (r = 0.15-0.27), so the windows include pre/post movement. Each rep needs start/stop segmentation before the turn speeds can be used.
- **Splits:** the official `ten_yd_split` is present whenever `forty` is.
- **Game tracking:** 22.7 M frames, about 114 k plays over 3 seasons (PRE/REG/POST). Speed p99.9 ≈ 9.4 yd/s, but the max reaches 16.9 yd/s, so game data needs glitch filtering in Phase 4.
- **Outcomes** (`player_career_successes`): career snaps by phase, games active/started, All-Pro, Pro Bowl. Very few honours (2.2% Pro Bowl, 1.2% 1st-team All-Pro), so the tenure models should rely on snaps and games.